# CBUS mini-reproduction: free Kaggle GPU runner (vLLM)  -  UNTESTED
Settings: Accelerator **GPU T4 x2**, **Internet ON**. No API key, no money. Works in an interactive session (do not rely on 'Save & Run All').
The authors' data, the vLLM log and the LLM cache stay in `/tmp` (licensed texts are not saved as output). Results (small CSVs) go to `/kaggle/working/cbus-repro/runs`.
Quota is limited (about 30 GPU-hours per week; check your Kaggle profile), so stop the session when you are done.

## 1. Setup

In [1]:
import os, subprocess
REPO = 'https://github.com/sarrazer24/cbus-repro'  
if not os.path.exists('/kaggle/working/cbus-repro'):
    subprocess.run(['git','clone',REPO,'/kaggle/working/cbus-repro'],check=True)
if not os.path.exists('/tmp/authors_data'):
    subprocess.run(['git','clone','--depth','1','https://github.com/iai-group/emnlp2026-simreading','/tmp/authors_data'],check=True)
os.chdir('/kaggle/working/cbus-repro')
os.environ['SIMREADING_DATA'] = '/tmp/authors_data'
subprocess.run('pip install -q -r requirements.txt',shell=True)
print(os.getcwd())

Cloning into '/kaggle/working/cbus-repro'...
Cloning into '/tmp/authors_data'...


/kaggle/working/cbus-repro


In [2]:
!nvidia-smi

Thu Nov 12 05:44:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install vLLM (can take several minutes; if pip complains about torch, restart the kernel once and re-run from the next cell)

In [3]:
!pip install -q vllm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


## 3. Start a local OpenAI-compatible server
Default: **Qwen2.5-7B-Instruct** in fp16, split over both T4s (one T4 has too little memory for 7B fp16). Fallback if that fails: set `MODEL='Qwen/Qwen2.5-3B-Instruct'` and `TP=1`.
If the server dies, read `/tmp/vllm.log` (next cell).

In [4]:
import subprocess, time, requests
MODEL = 'Qwen/Qwen2.5-7B-Instruct'; TP = 2; NAME = 'local'
log = open('/tmp/vllm.log','w')
env = dict(os.environ, VLLM_ATTENTION_BACKEND='XFORMERS')   # T4 has no FlashAttention-2
proc = subprocess.Popen(['python','-m','vllm.entrypoints.openai.api_server','--model',MODEL,'--served-model-name',NAME,
    '--dtype','half','--tensor-parallel-size',str(TP),'--max-model-len','8192','--gpu-memory-utilization','0.90',
    '--enable-prefix-caching','--port','8000'], stdout=log, stderr=subprocess.STDOUT, env=env)
for i in range(120):
    try:
        if requests.get('http://localhost:8000/v1/models',timeout=2).ok: print('server ready after',i*5,'s'); break
    except Exception: pass
    if proc.poll() is not None: print('server exited; see /tmp/vllm.log'); break
    time.sleep(5)

server ready after 295 s


In [5]:
!tail -n 30 /tmp/vllm.log

(APIServer pid=103) INFO 11-12 05:53:19 [launcher.py:197] Route: /detokenize, Methods: POST
(APIServer pid=103) INFO 11-12 05:53:19 [launcher.py:197] Route: /v1/models, Methods: GET
(APIServer pid=103) INFO 11-12 05:53:19 [launcher.py:197] Route: /v1/systemone, Methods: POST
(APIServer pid=103) INFO 11-12 05:53:19 [launcher.py:197] Route: /v1/decisions, Methods: POST
(APIServer pid=103) INFO 11-12 05:53:19 [launcher.py:197] Route: /ping, Methods: GET
(APIServer pid=103) INFO 11-12 05:53:19 [launcher.py:197] Route: /ping, Methods: POST
(APIServer pid=103) INFO 11-12 05:53:19 [launcher.py:197] Route: /invocations, Methods: POST
(APIServer pid=103) INFO 11-12 05:53:19 [launcher.py:197] Route: /v1/chat/completions, Methods: POST
(APIServer pid=103) INFO 11-12 05:53:19 [launcher.py:197] Route: /v1/chat/completions/batch, Methods: POST
(APIServer pid=103) INFO 11-12 05:53:19 [launcher.py:197] Route: /v1/responses, Methods: POST
(APIServer pid=103) INFO 11-12 05:53:19 [launcher.py:197] Route:

## 4. Smoke test and pilot (20 students). Inspect `meta.json` (drop rate!) and the table before scaling.

In [6]:
COMMON = '--base-url http://localhost:8000/v1 --model local --max-tokens 400 --workers 32 --cache /tmp/cache.jsonl'
!python scripts/02_run_simulation.py --mode baseline --students 1 --texts-per-student 1 {COMMON} --out-root /tmp/smoke

[run] baseline-c4-local-n1-s0: 1 students x ~1 texts
  1/1 student-texts
[done] answered=7 dropped=0 (0.00%) usage={'calls': 7, 'cache_hits': 0, 'input_tokens': 6856, 'output_tokens': 42} -> /tmp/smoke/baseline-c4-local-n1-s0


In [7]:
for mode in ['baseline','spr','ts']:
    !python scripts/02_run_simulation.py --mode {mode} --students 20 {COMMON} --out-root runs_pilot
!python scripts/03_evaluate.py "runs_pilot/*" --no-authors

[run] baseline-c4-local-n20-s0: 20 students x ~7 texts
  14/140 student-texts
  28/140 student-texts
  42/140 student-texts
  56/140 student-texts
  70/140 student-texts
  84/140 student-texts
  98/140 student-texts
  112/140 student-texts
  126/140 student-texts
  140/140 student-texts
[done] answered=680 dropped=0 (0.00%) usage={'calls': 680, 'cache_hits': 7, 'input_tokens': 614553, 'output_tokens': 4632} -> runs_pilot/baseline-c4-local-n20-s0
[run] spr-c4-local-n20-s0: 20 students x ~7 texts
  14/140 student-texts
  28/140 student-texts
  42/140 student-texts
  56/140 student-texts
  70/140 student-texts
  84/140 student-texts
  98/140 student-texts
  112/140 student-texts
  126/140 student-texts
  140/140 student-texts
[done] answered=670 dropped=10 (1.47%) usage={'calls': 810, 'cache_hits': 0, 'input_tokens': 313663, 'output_tokens': 17345} -> runs_pilot/spr-c4-local-n20-s0
[run] ts-c2-local-n20-s0: 20 students x ~7 texts
  14/140 student-texts
  28/140 student-texts
  42/140 stud

In [8]:
import json,glob
for f in sorted(glob.glob('runs_pilot/*/meta.json')):
    m=json.load(open(f)); print(m['name'], 'dropped_stage1', m['dropped_stage1_calls'], 'dropped_answers', m['dropped_answer_replies'], 'seconds', m['seconds'], m['usage'])
for l in list(open('/tmp/cache.jsonl',encoding='utf-8'))[:6]: print(json.loads(l)['v'][:200].replace('\n',' '),'\n---')

baseline-c4-local-n20-s0 dropped_stage1 0 dropped_answers 0 seconds 72.9 {'calls': 680, 'cache_hits': 7, 'input_tokens': 614553, 'output_tokens': 4632}
spr-c4-local-n20-s0 dropped_stage1 2 dropped_answers 0 seconds 109.2 {'calls': 810, 'cache_hits': 0, 'input_tokens': 313663, 'output_tokens': 17345}
ts-c2-local-n20-s0 dropped_stage1 10 dropped_answers 1 seconds 211.5 {'calls': 1350, 'cache_hits': 0, 'input_tokens': 855946, 'output_tokens': 38654}
{"answer": true} 
---
{"answer": false} 
---
{"answer": false} 
---
{"answer": true} 
---
{"answer": false} 
---
{"answer": false} 
---


## 5. Main runs (300 students). One cell per condition. Re-running a cell resumes from the cache while the session lives.

In [9]:
!python scripts/02_run_simulation.py --mode random --students 300
!python scripts/02_run_simulation.py --mode baseline --students 300 {COMMON}

[run] random-c4-none-n300-s0: 300 students x ~7 texts
  210/2100 student-texts
  420/2100 student-texts
  630/2100 student-texts
  840/2100 student-texts
  1050/2100 student-texts
  1260/2100 student-texts
  1470/2100 student-texts
  1680/2100 student-texts
  1890/2100 student-texts
  2100/2100 student-texts
[done] answered=10396 dropped=0 (0.00%) usage=None -> runs/random-c4-none-n300-s0
[run] baseline-c4-local-n300-s0: 300 students x ~7 texts
  210/2100 student-texts
  420/2100 student-texts
  630/2100 student-texts
  840/2100 student-texts
  1050/2100 student-texts
  1260/2100 student-texts
  1470/2100 student-texts
  1680/2100 student-texts
  1890/2100 student-texts
  2100/2100 student-texts
[done] answered=10396 dropped=0 (0.00%) usage={'calls': 10396, 'cache_hits': 680, 'input_tokens': 8818653, 'output_tokens': 66992} -> runs/baseline-c4-local-n300-s0


In [10]:
!python scripts/02_run_simulation.py --mode spr --students 300 {COMMON}

[run] spr-c4-local-n300-s0: 300 students x ~7 texts
  210/2100 student-texts
  420/2100 student-texts
  630/2100 student-texts
  840/2100 student-texts
  1050/2100 student-texts
  1260/2100 student-texts
  1470/2100 student-texts
  1680/2100 student-texts
  1890/2100 student-texts
  2100/2100 student-texts
[done] answered=10218 dropped=178 (1.71%) usage={'calls': 12331, 'cache_hits': 810, 'input_tokens': 4439202, 'output_tokens': 240099} -> runs/spr-c4-local-n300-s0


In [11]:
!python scripts/02_run_simulation.py --mode spr_dropout --students 300 {COMMON}

[run] spr_dropout-c4-local-n300-s0: 300 students x ~7 texts
  210/2100 student-texts
  420/2100 student-texts
  630/2100 student-texts
  840/2100 student-texts
  1050/2100 student-texts
  1260/2100 student-texts
  1470/2100 student-texts
  1680/2100 student-texts
  1890/2100 student-texts
  2100/2100 student-texts
[done] answered=9885 dropped=511 (4.92%) usage={'calls': 12030, 'cache_hits': 1, 'input_tokens': 4507927, 'output_tokens': 491866} -> runs/spr_dropout-c4-local-n300-s0


In [12]:
!python scripts/02_run_simulation.py --mode ts --students 300 {COMMON}

[run] ts-c2-local-n300-s0: 300 students x ~7 texts
  210/2100 student-texts
  420/2100 student-texts
  630/2100 student-texts
  840/2100 student-texts
  1050/2100 student-texts
  1260/2100 student-texts
  1470/2100 student-texts
  1680/2100 student-texts
  1890/2100 student-texts
  2100/2100 student-texts
[done] answered=10274 dropped=122 (1.17%) usage={'calls': 20683, 'cache_hits': 1350, 'input_tokens': 12214935, 'output_tokens': 574327} -> runs/ts-c2-local-n300-s0


## 6. Evaluate and export (download `outputs.zip` from the Output panel, then stop the session)

In [13]:
!python scripts/03_evaluate.py

             source                                       name  n_students  mean_acc  score_sd   gap   jsd   ece  pearson  spearman  n_responses  dropped_pct
            authors                Humans (test, ground truth)        1197     0.685     0.173   NaN   NaN   NaN      NaN       NaN          NaN          NaN
            authors                          Humans (held-out)        1161     0.661     0.181 0.026 0.006 0.027    0.895     0.845      35393.0          NaN
authors (mean of 3)                                     random        1198     0.378     0.121 0.309 0.518 0.306    0.409     0.319      35686.0          NaN
authors (mean of 3)        llm_baseline-llama-3.3-70b-instruct        1198     0.966     0.036 0.279 0.679 0.285    0.241     0.179      35686.0          NaN
authors (mean of 3)          cbus_ts-c2-llama-3.3-70b-instruct        1198     0.925     0.068 0.238 0.475 0.244    0.455     0.304      35686.0          NaN
authors (mean of 3)         cbus_spr-c4-llama-3.3-70

In [14]:
import shutil
shutil.make_archive('/kaggle/working/outputs','zip','.','runs'); shutil.make_archive('/kaggle/working/results','zip','.','results')
proc.terminate()
for p in ['/tmp/cache.jsonl','/tmp/authors_data','/tmp/smoke','runs_pilot']:
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else (os.path.exists(p) and os.remove(p))
print('exported; cache and authors data removed')

exported; cache and authors data removed
